# Modelagem Preditiva — Home Credit Credit Risk Model Stability


---

## 1.0 Configuracao e Carregamento da Master Table

**O que faz:** inicializa o ambiente, importa funcoes de `utils_2.py` e carrega a `master_table.parquet` exportada pelo notebook `02_Feature_Engineering`.

**Por que importa:** garante que o pipeline de modelagem consome exatamente a mesma base utilizada no feature engineering, sem perda de colunas ou granularidade.

**Como usar:** execute uma vez por sessao.

In [5]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Aponta o Python para a pasta do projeto ANTES de importar
project_dir = '/content/Projeto_06'
if project_dir not in sys.path:
    sys.path.append(project_dir)

os.chdir(project_dir)

# 2. Agora o Python já sabe onde encontrar o utils_2.py
from utils_2 import (
    setup_environment,
    avaliar_classificador,
    avaliar_classificacao_oot,
)

# 3. Configura o ambiente e lê os dados
PROJECT_ROOT = setup_environment()


Ambiente configurado. Token Kaggle carregado.


---

## 3.0 Baseline de Modelos (Cross-Validation)

**O que faz:** treina 5 modelos baseline com Stratified K-Fold (CV=3) e avalia metricas independentes (ROC-AUC, PR-AUC, KS) com Intervalo de Confianca 95% corrigido.

**Por que importa:** o baseline estabelece o piso de performance. O gap entre treino e validacao indica overfitting. O IC 95% quantifica a incerteza das metricas.

**Como ler:** Gap (V-T) negativo indica overfitting. ICs largos indicam alta variancia entre folds (instabilidade).

In [6]:
output_dir = PROJECT_ROOT / 'data' / 'processed'

# 1. Leitura dos arquivos Parquet
df_train = pd.read_parquet(output_dir / 'df_train.parquet', engine='pyarrow')
df_test = pd.read_parquet(output_dir / 'df_test.parquet', engine='pyarrow')
df_oot = pd.read_parquet(output_dir / 'df_oot.parquet', engine='pyarrow')

# 2. Identificação de colunas que NÃO podem entrar na matriz de treino
colunas_ignorar = ['target', 'case_id', 'date_decision', 'MONTH', 'WEEK_NUM']
features_cols = [c for c in df_train.columns if c not in colunas_ignorar]

# 3. Matrizes de Features (X) e Vetores Target (y)
X_train = df_train[features_cols]
y_train = df_train['target'].astype('int8')

X_test = df_test[features_cols]
y_test = df_test['target'].astype('int8')

X_oot = df_oot[features_cols]
y_oot = df_oot['target'].astype('int8')

# 4. Auditoria de dimensões e taxa de inadimplência
print(f"X_train shape: {X_train.shape} | Default: {y_train.mean()*100:.2f}%")
print(f"X_test shape:  {X_test.shape}  | Default: {y_test.mean()*100:.2f}%")
print(f"X_oot shape:   {X_oot.shape}   | Default: {y_oot.mean()*100:.2f}%")
print(f"Features preditoras: {len(features_cols)}")

X_train shape: (963493, 48) | Default: 3.26%
X_test shape:  (412926, 48)  | Default: 3.26%
X_oot shape:   (150240, 48)   | Default: 2.11%
Features preditoras: 48


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Cálculo do desbalanceamento (garantindo que é inteiro para o XGBoost)
taxa_desbalanceamento = int(y_train.value_counts()[0] / y_train.value_counts()[1])
print(f"Taxa de desbalanceamento: {taxa_desbalanceamento}:1")

modelos_baseline = {
    'Logistic_Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')), #mediana para o imputer de nulos
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1
        ))
    ]),
    
    'Random_Forest': Pipeline([
        # Imputação com -9999 ajuda a árvore a criar um "nó" exclusivo para NaNs 
        ('imputer', SimpleImputer(strategy='constant', fill_value=-9999)), #nó para imputer de nulos
        ('model', RandomForestClassifier(
            n_estimators=100, max_depth=10, class_weight='balanced',
            random_state=42, n_jobs=-1
        ))
    ]),
    
    # LightGBM lida com NaNs nativamente
    'LightGBM': LGBMClassifier(
        class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1 
    ),
    
    # XGBoost lida com NaNs nativamente. Acelerado via GPU (cuda).
    'XGBoost': XGBClassifier(
        scale_pos_weight=taxa_desbalanceamento, 
        eval_metric='auc',
        n_estimators=100, 
        random_state=42, 
        tree_method='hist', 
        device='cuda' 
    ),
    
    # CatBoost lida com NaNs nativamente.
    'CatBoost': CatBoostClassifier(
        auto_class_weights='Balanced', 
        eval_metric='AUC', 
        iterations=100,
        random_seed=42, 
        verbose=False, 
        thread_count=-1  # Usa todos os núcleos da CPU
    )
}

# Dicionário de thresholds iniciais focado em 0.5 para todos os modelos
thresholds_baseline = {
    'Logistic_Regression': 0.5,
    'Random_Forest': 0.5,
    'LightGBM': 0.5,
    'XGBoost': 0.5,
    'CatBoost': 0.5
}

modelos_ajustados = {}

for nome, modelo in modelos_baseline.items():
    modelos_ajustados[nome] = avaliar_classificador(
        modelo=modelo,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        nome_modelo=nome,
        cv=3,
        thresholds_dict=thresholds_baseline  # Dicionário mapeado aqui
    )

Taxa de desbalanceamento: 29:1
 LOGISTIC_REGRESSION (CV = 3 | Threshold = 0.5000)
---------------------------------------------------------------------------------------------------------
 Taxa Base de Default (Target=1): Treino = 3.26% | Teste = 3.26%
---------------------------------------------------------------------------------------------------------

 [ METRICAS INDEPENDENTES DO THRESHOLD (Probabilidades) ]
 Metrica    | Treino     | Teste      | Gap(Te-Tr) | Media CV (Treino)  | IC 95% (CV Treino)
--------------------------------------------------------------------------------------------
 ROC-AUC    | 0.7184     | 0.7202     | +0.0019    | 0.7176             | [0.7161, 0.7192]
 GINI       | 0.4367     | 0.4405     | +0.0038    | 0.4353             | [0.4322, 0.4383]
 PR-AUC     | 0.0843     | 0.0835     | -0.0008    | 0.0842             | -
 KS         | 0.3321     | 0.3370     | +0.0050    | 0.3316             | -

 [ METRICAS DEPENDENTES DO THRESHOLD (Corte = 0.5000) ]
 Metr